# ISOM 839 · Session 5 — Network Optimization II: Flows, Paths & Supply Chain Design

**Prescriptive Analytics: Modeling & Optimization · Suffolk University · Prof. Hasan Arslan**

Last week every node was a plant, a hub, or a warehouse. Tonight the same skeleton — **flow in, flow out, cost per arc** — answers three different questions: the fastest route (shortest path), the most we can push through (max flow), and the cheapest way to meet every demand under capacity (min-cost flow). Amazon's regionalization and Microsoft's Edelman-winning fulfillment engine are the third one, at scale.

In [ ]:
%pip install -q gurobipy
import gurobipy as gp
from gurobipy import GRB
print('gurobipy ready:', gp.gurobi.version())

---
## Part 1 — Shortest path: send one unit, pay per minute

A courier leaves the **Depot** for a **Client** across Boston. Arcs are road segments; costs are minutes. Shortest path is a min-cost flow where the depot supplies one unit and the client demands one: every node in between passes it along.

**Node balance** is the one constraint family: *out − in = +1 at the source, −1 at the sink, 0 everywhere else.*

In [ ]:
minutes = {('Depot','A'): 12, ('Depot','B'): 9, ('A','C'): 7, ('B','A'): 4, ('B','D'): 15,
           ('C','D'): 6, ('C','Client'): 14, ('D','Client'): 5}
nodes = {n for arc in minutes for n in arc}

sp = gp.Model('shortest_path'); sp.Params.OutputFlag = 0
use = sp.addVars(minutes.keys(), name='use')                      # continuous — integrality comes free
sp.setObjective(use.prod(minutes), GRB.MINIMIZE)
for n in nodes:
    rhs = 1 if n == 'Depot' else (-1 if n == 'Client' else 0)
    sp.addConstr(use.sum(n, '*') - use.sum('*', n) == rhs, name=f'balance_{n}')
sp.optimize()

print(f'Fastest route: {sp.ObjVal:.0f} minutes')
route = [arc for arc, v in use.items() if v.X > 0.5]
print('  arcs used:', route)

Depot → B → D → Client in 29 minutes. Depot → A looks tempting at 12 but A's only exits are slow. **The model never looks at one arc at a time — and neither should you.**

### 💬 Discussion
Dijkstra solves this in microseconds without a solver. Why would anyone write it as an LP? *(Hint: what happens when the courier also has a capacity, or a second client, or a cost per stop?)*

---
## Part 2 — Max flow: how much can the network carry?

A regional pipeline from source **S** to terminal **T**. Numbers are capacities in units per hour; there is no cost. The objective becomes *maximize what leaves S*, and balance at every interior node says nothing leaks.

In [ ]:
capacity = {('S','A'): 40, ('S','B'): 30, ('A','B'): 15, ('A','C'): 10, ('A','T'): 25, ('B','C'): 35, ('C','T'): 30}

mf = gp.Model('max_flow'); mf.Params.OutputFlag = 0
flow = mf.addVars(capacity.keys(), ub=capacity, name='flow')       # ub = arc capacity
mf.setObjective(flow.sum('S', '*'), GRB.MAXIMIZE)
for n in ['A', 'B', 'C']:
    mf.addConstr(flow.sum('*', n) == flow.sum(n, '*'), name=f'balance_{n}')
mf.optimize()

print(f'Maximum throughput: {mf.ObjVal:.0f} units/hour')
for arc, v in flow.items():
    tag = '  <- FULL' if abs(v.X - capacity[arc]) < 1e-6 else ''
    print(f'  {arc[0]} -> {arc[1]}: {v.X:4.0f} / {capacity[arc]}{tag}')

55 units an hour, and look at which arcs are full: **A→T (25) and C→T (30)** — every unit that reaches T crosses one of them, and 25 + 30 = 55. That set of full arcs is the **min cut**. Max flow equals min cut, always; the bottleneck is a *set* of arcs, not one.

**Try:** raise `('C','T')` to 40. Does throughput rise by 10? Why not?

---
## Part 3 — Min-cost flow: the workhorse (tonight's lab)

**Your team is the supply chain analytics group for a mid-size retailer.** Two plants (Reading PA, Syracuse NY), three distribution centers (Worcester, Hartford, Albany), four demand regions. Every lane has a cost per unit and a weekly capacity. The question executives actually ask: *what is the cheapest way to meet every region's demand, and which lanes are holding us back?*

**Step 1 — sketch it before you code it.** Plants supply, regions demand, DCs pass through: at every DC, *flow in = flow out*. That one equation is the whole difference from last week's transportation model.

In [ ]:
plants  = {'Reading': 700, 'Syracuse': 500}                     # weekly supply
dcs     = ['Worcester', 'Hartford', 'Albany']                   # pure pass-through
regions = {'Boston': 420, 'Providence': 230, 'Manchester': 180, 'New Haven': 270}   # weekly demand

lane = {  # (cost $/unit, capacity units/week)
    ('Reading','Worcester'):  (5.1, 500), ('Reading','Hartford'):  (3.6, 500), ('Reading','Albany'):  (4.4, 400),
    ('Syracuse','Worcester'): (4.9, 350), ('Syracuse','Hartford'): (5.3, 300), ('Syracuse','Albany'): (2.2, 450),
    ('Worcester','Boston'): (1.6, 450), ('Worcester','Providence'): (2.9, 250), ('Worcester','Manchester'): (2.4, 250), ('Worcester','New Haven'): (4.7, 200),
    ('Hartford','Boston'):  (3.8, 300), ('Hartford','Providence'):  (2.7, 300), ('Hartford','Manchester'):  (5.6, 150), ('Hartford','New Haven'):  (1.5, 300),
    ('Albany','Boston'):    (5.9, 300), ('Albany','Providence'):    (6.4, 200), ('Albany','Manchester'):    (4.1, 250), ('Albany','New Haven'):    (5.2, 200),
}
cost = {k: v[0] for k, v in lane.items()}
cap  = {k: v[1] for k, v in lane.items()}
print(f'supply {sum(plants.values())} vs demand {sum(regions.values())}  ·  {len(lane)} lanes')

**Step 2 — one variable per lane, three constraint families plus capacity.** We write capacity as an explicit constraint (not a variable bound) so it carries a shadow price we can read.

In [ ]:
net = gp.Model('network_design'); net.Params.OutputFlag = 0
f = net.addVars(lane.keys(), name='f')
net.setObjective(f.prod(cost), GRB.MINIMIZE)

supply  = net.addConstrs((f.sum(p, '*') <= plants[p]  for p in plants),  'supply')    # plants ship at most their supply
balance = net.addConstrs((f.sum('*', d) == f.sum(d, '*') for d in dcs), 'balance')   # DCs: in = out
demand  = net.addConstrs((f.sum('*', r) == regions[r] for r in regions), 'demand')    # regions get exactly their demand
lanecap = net.addConstrs((f[k] <= cap[k] for k in lane), 'cap')                       # lane capacity, as a constraint
net.optimize()

print(f'Weekly cost: ${net.ObjVal:,.0f}\n')
for k, v in f.items():
    if v.X > 1e-6:
        tag = f'   FULL  (one more unit of capacity worth ${abs(lanecap[k].Pi):.2f})' if abs(v.X - cap[k]) < 1e-6 else ''
        print(f'  {k[0]:9s} -> {k[1]:10s} {v.X:5.0f} / {cap[k]}{tag}')
print('\nDC throughput:', {d: round(sum(f[k].X for k in lane if k[0] == d)) for d in dcs})

**Step 3 — read it like a manager.** Hartford carries 500 of the 1,100 units and Reading→Hartford runs full — yet its shadow price is $0.00, because a parallel route costs exactly the same at the margin (a *degenerate* binding constraint; you will meet one in the homework). Albany barely matters this week. Now the number executives pay for: the marginal cost to serve one more unit in each region.

In [ ]:
print('One more unit of demand would cost:')
for r in regions:
    print(f'  {r:11s} ${demand[r].Pi:.2f}')
print('\nOne more unit of plant capacity would save:')
for p in plants:
    print(f'  {p:9s} ${abs(supply[p].Pi):.2f}   (idle {supply[p].Slack:.0f})')

Boston is the most expensive region to grow at the margin even though its cheapest lane is $1.60 — the cheap Worcester lane is spoken for, so the next Boston unit cascades through the network. **A shadow price is a system number, not a lane number** (you heard that last week; now you have seen it twice).

**Step 4 — stress test: regionalization in miniature.** A carrier strike cuts the Worcester→Boston lane to 60% of capacity. Re-solve and watch the flow re-route.

In [ ]:
lanecap[('Worcester','Boston')].RHS = int(cap[('Worcester','Boston')] * 0.6)      # 450 -> 270
net.optimize()
print(f'Worcester->Boston at 60%: ${net.ObjVal:,.0f} per week')
for k, v in f.items():
    if v.X > 1e-6 and k[1] == 'Boston':
        print(f'  {k[0]:9s} -> Boston {v.X:5.0f}')
print('DC throughput:', {d: round(sum(f[k].X for k in lane if k[0] == d)) for d in dcs})

lanecap[('Worcester','Boston')].RHS = cap[('Worcester','Boston')]                 # put it back
net.optimize()

The 150 units Worcester can no longer deliver to Boston come from **Albany at $5.90** — not from Hartford at $3.80, because Hartford's inbound lane from Reading is already full. Cost rises $240 a week, and Albany, which barely mattered an hour ago, now carries 330 units. **The model re-plans the whole network, not the broken lane.** That re-planning, run nightly over hundreds of nodes, is what Amazon's regionalization and Microsoft's fulfillment engine actually are.

### 💬 Discussion
Which single lane would *you* expand first, and what would you pay per unit of capacity? Say it before you look at the shadow prices again.

---
## Part 4 — From tonight's toy to Amazon's network

What changes at scale: thousands of nodes, time periods, and the choice of *which* facilities exist (next week — that needs binary variables). What does not change: the three constraint families you just wrote. Network LPs with millions of arcs solve in minutes because the constraint matrix is sparse and totally unimodular — whole supplies in, whole shipments out.

---
# Homework #4 — Min-Cost Flow (Three-Echelon Network) 🚚

**On Canvas: this notebook completed + a one-paragraph memo in the text box.** Due date and rubric on the Canvas assignment; late window per the syllabus.

**Keystone Distribution** moves product from two plants through three cross-docks to five demand zones each week. Every lane has a cost per unit and a weekly capacity; nothing accumulates at a cross-dock.

| Plant | Supply | | Zone | Demand |
|---|---|---|---|---|
| Allentown | 900 | | Boston | 500 |
| Albany | 700 | | Providence | 280 |
| | | | Manchester | 220 |
| | | | Portland | 240 |
| | | | New Haven | 300 |

Lanes (cost $/unit, capacity units/week) are in the code cell below.

1. **Formulate** the model in a markdown cell: flow variables, cost objective, node balance at plants, cross-docks and zones, and lane capacities.
2. **Build and solve** it in gurobipy. Print the optimal flow plan (every lane that carries flow) and total weekly cost.
3. **The two most binding lane capacities**, with their shadow prices and one sentence each on what the number is worth to the business.
4. **One-paragraph memo** to the VP of Supply Chain: which *single* lane to expand, why, and roughly what a unit of extra capacity there is worth. Cite the shadow price. No jargon.

In [ ]:
hw_plants = {'Allentown': 900, 'Albany': 700}
hw_docks  = ['Worcester', 'Hartford', 'Newark']
hw_zones  = {'Boston': 500, 'Providence': 280, 'Manchester': 220, 'Portland': 240, 'New Haven': 300}
hw_lane = {  # (cost $/unit, capacity units/week)
    ('Allentown','Worcester'): (6.5, 500), ('Allentown','Hartford'): (4.2, 600), ('Allentown','Newark'): (2.8, 400),
    ('Albany','Worcester'):    (3.1, 420), ('Albany','Hartford'):    (4.6, 300), ('Albany','Newark'):    (6.9, 350),
    ('Worcester','Boston'): (1.9, 380), ('Worcester','Providence'): (3.4, 250), ('Worcester','Manchester'): (2.7, 300), ('Worcester','Portland'): (4.8, 200),
    ('Hartford','Boston'):  (3.9, 350), ('Hartford','Providence'):  (2.6, 300), ('Hartford','New Haven'):  (1.8, 220), ('Hartford','Portland'):  (6.4, 150),
    ('Newark','New Haven'): (3.2, 300), ('Newark','Providence'):    (5.7, 200), ('Newark','Boston'):      (7.1, 300),
}
hw_cost = {k: v[0] for k, v in hw_lane.items()}
hw_cap  = {k: v[1] for k, v in hw_lane.items()}

hw = gp.Model('keystone'); hw.Params.OutputFlag = 0

# TODO 1: one continuous flow variable per lane

# TODO 2: objective — minimize total weekly cost

# TODO 3: node balance — plants ship <= supply; cross-docks in == out; zones receive == demand

# TODO 4: lane capacities as named constraints (so you can read their .Pi)

hw.optimize()

# TODO 5: print the flow plan and total cost

# TODO 6: list the binding lanes with their shadow prices; pick the two that matter most

# TODO 7: write the memo paragraph in a markdown cell below (and paste it into the Canvas text box)

---
### Submission checklist
- [ ] Formulation written in a markdown cell — all three node types named
- [ ] Model solves; flow plan and total cost printed
- [ ] Two binding lanes identified with shadow prices (a number each, and a sentence each)
- [ ] Memo paragraph: one lane, one reason, one dollar figure — pasted into the Canvas text box

**Next week — Session 6:** the network stops being a given. Open the warehouse or don't — binary variables, fixed costs, and why one `vtype` change makes the model a strategist. 🏗️